# Structure Prediction for FireProtDB Mutants

**Objective:** Obtain structures for missing WT proteins and all 72 mutants

**Strategy:**
1. Check AlphaFold Database for missing WT structures (P00720, P69543)
2. Compare AlphaFold2 vs Boltz-2 on test cases (1 WT, 1 mutant)
3. Batch predict all 72 mutants with chosen tool

**Author:** Shyam Chandra  
**Date:** October 17, 2025

**Note:** Run on Google Colab with GPU for optimal performance

## Setup and Data Loading

In [1]:
import pandas as pd
import requests
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt

# Load selected mutants
mutants_df = pd.read_csv("mit_media_lab_selected_mutants.csv")

print(f"Loaded {len(mutants_df)} mutants across {mutants_df['uniprot_id'].nunique()} proteins")

# Identify missing WT structures
missing_wt = mutants_df[mutants_df['wt_structure_path'] == 'NEEDS_GENERATION']['uniprot_id'].unique()
print(f"\nMissing WT structures: {', '.join(missing_wt)}")

Loaded 72 mutants across 12 proteins

Missing WT structures: P00720, P69543


## Step 1: Check Structure Databases for Missing WTs

Check both AlphaFold Database and ESMFold Atlas for pre-computed structures.

In [2]:
def check_alphafold_db(uniprot_id):
    """Check AlphaFold Database for pre-computed structure."""
    base_url = "https://alphafold.ebi.ac.uk/files"
    
    for version in ['v4', 'v3', 'v2']:
        pdb_url = f"{base_url}/AF-{uniprot_id}-F1-model_{version}.pdb"
        try:
            response = requests.head(pdb_url, timeout=10)
            if response.status_code == 200:
                return {'exists': True, 'url': pdb_url, 'version': version, 'source': 'AlphaFold'}
        except:
            continue
    return {'exists': False, 'url': None}

def check_esmfold_atlas(sequence):
    """Check ESMFold Atlas for pre-computed structure by sequence."""
    api_url = "https://api.esmatlas.com/fetchPredictedStructure"
    
    try:
        response = requests.post(api_url, json={'sequence': sequence}, timeout=10)
        if response.status_code == 200:
            return {'exists': True, 'data': response.text, 'source': 'ESMFold'}
    except:
        pass
    return {'exists': False, 'data': None}

def download_structure(uniprot_id, sequence=None, save_dir="../707_Files_for_Colab/WT_PDBs"):
    """Download structure from AlphaFold DB or ESMFold Atlas if available."""
    save_path = Path(save_dir) / f"{uniprot_id}.pdb"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Try AlphaFold Database first
    af_result = check_alphafold_db(uniprot_id)
    if af_result['exists']:
        response = requests.get(af_result['url'])
        with open(save_path, 'w') as f:
            f.write(response.text)
        print(f"{uniprot_id}: Downloaded from AlphaFold DB ({af_result['version']})")
        return save_path
    
    # Try ESMFold Atlas if sequence provided
    if sequence:
        esm_result = check_esmfold_atlas(sequence)
        if esm_result['exists']:
            with open(save_path, 'w') as f:
                f.write(esm_result['data'])
            print(f"{uniprot_id}: Downloaded from ESMFold Atlas")
            return save_path
    
    print(f"{uniprot_id}: Not found in databases (will need prediction)")
    return None

# Check and download missing WTs
print("Checking structure databases...\n")
for uniprot_id in missing_wt:
    # Get sequence for ESMFold check
    seq = mutants_df[mutants_df['uniprot_id'] == uniprot_id].iloc[0]['sequence']
    download_structure(uniprot_id, sequence=seq)

Checking structure databases...

P00720: Not found in databases (will need prediction)
P00720: Not found in databases (will need prediction)
P69543: Not found in databases (will need prediction)
P69543: Not found in databases (will need prediction)


## Step 2: Install Prediction Tools (Colab Only - Skip Locally)

Run this cell only on Google Colab with GPU. Skip if running locally.

In [ ]:
# Install ColabFold for AlphaFold2
!pip install -q colabfold[alphafold]

# Install Boltz-2
!pip install -q boltz

print("Installation complete")

## Step 3: Define Prediction Functions

In [ ]:
def predict_alphafold2(sequence, output_path, name="protein"):
    """
    Predict structure using ColabFold (AlphaFold2).
    
    Args:
        sequence: Protein sequence
        output_path: Where to save PDB
        name: Protein name for output
    
    Returns:
        dict with pdb_path, plddt_scores, ptm_score, runtime
    """
    from colabfold.batch import run_prediction
    from colabfold.download import download_alphafold_params
    import tempfile
    
    start_time = time.time()
    
    # Setup temporary directory
    with tempfile.TemporaryDirectory() as tmpdir:
        # Write input FASTA
        fasta_path = Path(tmpdir) / f"{name}.fasta"
        with open(fasta_path, 'w') as f:
            f.write(f">{name}\n{sequence}\n")
        
        # Run prediction
        run_prediction(
            input_path=str(fasta_path),
            output_path=tmpdir,
            num_models=1,  # Use only model_1 for speed
            num_recycle=3,  # Default recycling
            use_gpu_relax=False,  # Skip AMBER relaxation for speed
        )
        
        # Find output PDB
        pdb_files = list(Path(tmpdir).glob("*.pdb"))
        if not pdb_files:
            return None
        
        # Copy to output location
        import shutil
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(pdb_files[0], output_path)
        
        # Extract pLDDT from B-factor column
        plddt_scores = []
        with open(output_path) as f:
            for line in f:
                if line.startswith("ATOM"):
                    plddt = float(line[60:66].strip())
                    plddt_scores.append(plddt)
        
        # Read pTM score from log (simplified - actual implementation may vary)
        ptm_score = np.mean(plddt_scores) / 100.0  # Approximation
    
    runtime = time.time() - start_time
    
    return {
        'pdb_path': str(output_path),
        'plddt_scores': plddt_scores,
        'mean_plddt': np.mean(plddt_scores),
        'ptm_score': ptm_score,
        'runtime': runtime
    }

def predict_boltz2(sequence, output_path, name="protein"):
    """
    Predict structure using Boltz-2.
    
    Returns:
        dict with pdb_path, plddt_scores, ptm_score, runtime
    """
    from boltz.main import Boltz2Model
    
    start_time = time.time()
    
    # Initialize model
    model = Boltz2Model()
    
    # Predict
    result = model.predict(sequence)
    
    # Save PDB
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.save_pdb(str(output_path))
    
    # Extract metrics
    plddt_scores = result.plddt if hasattr(result, 'plddt') else []
    ptm_score = result.ptm if hasattr(result, 'ptm') else None
    
    runtime = time.time() - start_time
    
    return {
        'pdb_path': str(output_path),
        'plddt_scores': plddt_scores,
        'mean_plddt': np.mean(plddt_scores) if plddt_scores else None,
        'ptm_score': ptm_score,
        'runtime': runtime
    }

print("Prediction functions defined")

## Step 4: Tool Comparison - Test Cases

Compare AlphaFold2 vs Boltz-2 on:
1. One missing WT protein
2. One mutant

In [ ]:
# Test case 1: Missing WT protein
if len(missing_wt) > 0:
    test_wt_id = missing_wt[0]
    test_wt_row = mutants_df[mutants_df['uniprot_id'] == test_wt_id].iloc[0]
    test_wt_seq = test_wt_row['sequence']
    
    print(f"Test WT: {test_wt_row['protein_name']} ({test_wt_id})")
    print(f"Sequence length: {len(test_wt_seq)} aa\n")
else:
    # Use existing WT for comparison
    test_wt_row = mutants_df.iloc[0]
    test_wt_id = test_wt_row['uniprot_id']
    test_wt_seq = test_wt_row['sequence']
    print(f"Using existing WT for test: {test_wt_row['protein_name']} ({test_wt_id})")
    print(f"Sequence length: {len(test_wt_seq)} aa\n")

# Test case 2: First mutant
test_mut_row = mutants_df.iloc[0]
test_mut_seq = test_mut_row['mt_sequence']
test_mut_name = f"{test_mut_row['uniprot_id']}_{test_mut_row['mutation_notation']}"

print(f"Test mutant: {test_mut_row['protein_name']} {test_mut_row['mutation_notation']}")
print(f"Sequence length: {len(test_mut_seq)} aa")

### Predict WT with Both Tools

In [ ]:
print("Predicting WT structure with AlphaFold2...")
af2_wt_result = predict_alphafold2(
    test_wt_seq,
    f"test_structures/{test_wt_id}_AF2.pdb",
    name=test_wt_id
)

print(f"\nAlphaFold2 WT Results:")
print(f"  Mean pLDDT: {af2_wt_result['mean_plddt']:.2f}")
print(f"  pTM score: {af2_wt_result['ptm_score']:.3f}")
print(f"  Runtime: {af2_wt_result['runtime']:.1f}s")

print("\n" + "="*60)

print("\nPredicting WT structure with Boltz-2...")
boltz_wt_result = predict_boltz2(
    test_wt_seq,
    f"test_structures/{test_wt_id}_Boltz2.pdb",
    name=test_wt_id
)

print(f"\nBoltz-2 WT Results:")
print(f"  Mean pLDDT: {boltz_wt_result['mean_plddt']:.2f}")
print(f"  pTM score: {boltz_wt_result['ptm_score']:.3f}")
print(f"  Runtime: {boltz_wt_result['runtime']:.1f}s")

### Predict Mutant with Both Tools

In [ ]:
print("Predicting mutant structure with AlphaFold2...")
af2_mut_result = predict_alphafold2(
    test_mut_seq,
    f"test_structures/{test_mut_name}_AF2.pdb",
    name=test_mut_name
)

print(f"\nAlphaFold2 Mutant Results:")
print(f"  Mean pLDDT: {af2_mut_result['mean_plddt']:.2f}")
print(f"  pTM score: {af2_mut_result['ptm_score']:.3f}")
print(f"  Runtime: {af2_mut_result['runtime']:.1f}s")

print("\n" + "="*60)

print("\nPredicting mutant structure with Boltz-2...")
boltz_mut_result = predict_boltz2(
    test_mut_seq,
    f"test_structures/{test_mut_name}_Boltz2.pdb",
    name=test_mut_name
)

print(f"\nBoltz-2 Mutant Results:")
print(f"  Mean pLDDT: {boltz_mut_result['mean_plddt']:.2f}")
print(f"  pTM score: {boltz_mut_result['ptm_score']:.3f}")
print(f"  Runtime: {boltz_mut_result['runtime']:.1f}s")

### Visualize pLDDT Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# WT pLDDT comparison
axes[0, 0].plot(af2_wt_result['plddt_scores'], label='AlphaFold2', alpha=0.7)
axes[0, 0].plot(boltz_wt_result['plddt_scores'], label='Boltz-2', alpha=0.7)
axes[0, 0].axhline(70, color='r', linestyle='--', alpha=0.3, label='Confidence threshold')
axes[0, 0].set_xlabel('Residue')
axes[0, 0].set_ylabel('pLDDT')
axes[0, 0].set_title(f'WT ({test_wt_id}) pLDDT Scores')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Mutant pLDDT comparison
axes[0, 1].plot(af2_mut_result['plddt_scores'], label='AlphaFold2', alpha=0.7)
axes[0, 1].plot(boltz_mut_result['plddt_scores'], label='Boltz-2', alpha=0.7)
axes[0, 1].axhline(70, color='r', linestyle='--', alpha=0.3, label='Confidence threshold')
axes[0, 1].set_xlabel('Residue')
axes[0, 1].set_ylabel('pLDDT')
axes[0, 1].set_title(f'Mutant ({test_mut_name}) pLDDT Scores')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Mean pLDDT comparison
tools = ['AlphaFold2', 'Boltz-2']
wt_means = [af2_wt_result['mean_plddt'], boltz_wt_result['mean_plddt']]
mut_means = [af2_mut_result['mean_plddt'], boltz_mut_result['mean_plddt']]

x = np.arange(len(tools))
width = 0.35

axes[1, 0].bar(x - width/2, wt_means, width, label='WT', alpha=0.7)
axes[1, 0].bar(x + width/2, mut_means, width, label='Mutant', alpha=0.7)
axes[1, 0].set_ylabel('Mean pLDDT')
axes[1, 0].set_title('Mean pLDDT Comparison')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(tools)
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Runtime comparison
wt_times = [af2_wt_result['runtime'], boltz_wt_result['runtime']]
mut_times = [af2_mut_result['runtime'], boltz_mut_result['runtime']]

axes[1, 1].bar(x - width/2, wt_times, width, label='WT', alpha=0.7)
axes[1, 1].bar(x + width/2, mut_times, width, label='Mutant', alpha=0.7)
axes[1, 1].set_ylabel('Runtime (seconds)')
axes[1, 1].set_title('Prediction Speed Comparison')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(tools)
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('tool_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nComparison plot saved to tool_comparison.png")

### Summary Metrics

**Interpretation Guide:**
- **pLDDT >90:** Very high confidence
- **pLDDT 70-90:** Good confidence
- **pLDDT <70:** Low confidence
- **pTM >0.8:** High confidence in overall fold
- **Speed:** Consider for 72-structure batch

**Next Steps:**
1. Review metrics above
2. Load PDBs in PyMOL: align and check RMSD
3. Choose tool based on accuracy/speed tradeoff
4. Proceed to batch prediction

In [ ]:
# Print summary table
summary = pd.DataFrame([
    {'Test': 'WT', 'Tool': 'AlphaFold2', 'Mean pLDDT': af2_wt_result['mean_plddt'], 
     'pTM': af2_wt_result['ptm_score'], 'Runtime (s)': af2_wt_result['runtime']},
    {'Test': 'WT', 'Tool': 'Boltz-2', 'Mean pLDDT': boltz_wt_result['mean_plddt'], 
     'pTM': boltz_wt_result['ptm_score'], 'Runtime (s)': boltz_wt_result['runtime']},
    {'Test': 'Mutant', 'Tool': 'AlphaFold2', 'Mean pLDDT': af2_mut_result['mean_plddt'], 
     'pTM': af2_mut_result['ptm_score'], 'Runtime (s)': af2_mut_result['runtime']},
    {'Test': 'Mutant', 'Tool': 'Boltz-2', 'Mean pLDDT': boltz_mut_result['mean_plddt'], 
     'pTM': boltz_mut_result['ptm_score'], 'Runtime (s)': boltz_mut_result['runtime']},
])

print("\nCOMPARISON SUMMARY")
print("="*80)
print(summary.to_string(index=False))
print("="*80)

# Projected time for 72 mutants
af2_total = (af2_wt_result['runtime'] + af2_mut_result['runtime']) / 2 * 72 / 3600
boltz_total = (boltz_wt_result['runtime'] + boltz_mut_result['runtime']) / 2 * 72 / 3600

print(f"\nProjected time for 72 mutants:")
print(f"  AlphaFold2: {af2_total:.1f} hours")
print(f"  Boltz-2: {boltz_total:.1f} hours")
print(f"\nWith Colab Pro: Can run overnight without disconnection")

## Step 5: Batch Prediction (Run After Choosing Tool)

Modify `chosen_tool` below based on comparison results.

In [ ]:
# SET THIS BASED ON COMPARISON
chosen_tool = 'alphafold2'  # or 'boltz2'

predict_fn = predict_alphafold2 if chosen_tool == 'alphafold2' else predict_boltz2

print(f"Using {chosen_tool.upper()} for batch prediction\n")

# Create output directory
output_dir = Path(f"predicted_structures_{chosen_tool}")
output_dir.mkdir(exist_ok=True)

# Track results
results = []

# Predict all mutants
for idx, row in mutants_df.iterrows():
    mutant_name = f"{row['uniprot_id']}_{row['mutation_notation']}"
    output_path = output_dir / f"{mutant_name}.pdb"
    
    print(f"[{idx+1}/{len(mutants_df)}] Predicting {mutant_name}...")
    
    result = predict_fn(
        row['mt_sequence'],
        output_path,
        name=mutant_name
    )
    
    results.append({
        'uniprot_id': row['uniprot_id'],
        'mutation': row['mutation_notation'],
        'protein_name': row['protein_name'],
        'ddG': row['ddG'],
        'stability_class': row['stability_class'],
        'mean_plddt': result['mean_plddt'],
        'ptm_score': result['ptm_score'],
        'runtime': result['runtime'],
        'pdb_path': result['pdb_path']
    })
    
    print(f"  pLDDT: {result['mean_plddt']:.2f}, Runtime: {result['runtime']:.1f}s\n")

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv(f"prediction_results_{chosen_tool}.csv", index=False)

print(f"\nBatch prediction complete!")
print(f"Results saved to prediction_results_{chosen_tool}.csv")
print(f"Structures saved to {output_dir}/")

## Step 6: Quality Control Summary

In [ ]:
# Analyze prediction quality
print("QUALITY CONTROL SUMMARY")
print("="*80)

high_conf = len(results_df[results_df['mean_plddt'] > 90])
good_conf = len(results_df[(results_df['mean_plddt'] >= 70) & (results_df['mean_plddt'] <= 90)])
low_conf = len(results_df[results_df['mean_plddt'] < 70])

print(f"\npLDDT Distribution:")
print(f"  High confidence (>90): {high_conf}/{len(results_df)} ({high_conf/len(results_df)*100:.1f}%)")
print(f"  Good confidence (70-90): {good_conf}/{len(results_df)} ({good_conf/len(results_df)*100:.1f}%)")
print(f"  Low confidence (<70): {low_conf}/{len(results_df)} ({low_conf/len(results_df)*100:.1f}%)")

print(f"\nMean pLDDT: {results_df['mean_plddt'].mean():.2f} ± {results_df['mean_plddt'].std():.2f}")
print(f"Mean pTM: {results_df['ptm_score'].mean():.3f} ± {results_df['ptm_score'].std():.3f}")
print(f"Total runtime: {results_df['runtime'].sum()/3600:.2f} hours")

# Plot quality distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(results_df['mean_plddt'], bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(70, color='r', linestyle='--', label='Confidence threshold')
axes[0].axvline(90, color='g', linestyle='--', label='High confidence')
axes[0].set_xlabel('Mean pLDDT')
axes[0].set_ylabel('Count')
axes[0].set_title('pLDDT Distribution Across All Predictions')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].scatter(results_df['ddG'], results_df['mean_plddt'], alpha=0.6)
axes[1].set_xlabel('ΔΔG (kcal/mol)')
axes[1].set_ylabel('Mean pLDDT')
axes[1].set_title('Structure Quality vs Stability Change')
axes[1].axhline(70, color='r', linestyle='--', alpha=0.3)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('batch_prediction_qc.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nQC plot saved to batch_prediction_qc.png")